# Build Constructor Standings

#### Sources
1. fact_session_results
1. dim_constructors

#### Output Columns
1. season
1. constructor id
1. constructor name
1. nationality
1. race starts
1. total points
1. number of wins
1. number of podiums
1. standing position

In [0]:
CREATE OR REPLACE VIEW formula1.gold.v_constructor_standing AS
with constructor_session_summary as (
  select
    r.season,
    c.constructor_id,
    c.constructor_name,
    c.nationality,
    count(*) as race_starts,
    sum(r.points) as total_points,
    count_if(r.is_win) as number_of_wins,
    count_if(r.is_podium) as number_of_podiums
  from
    formula1.gold.fact_session_results r
      join formula1.gold.dim_constructors c
        on r.constructor_id = c.constructor_id
  group by
    r.season,
    c.constructor_id,
    c.constructor_name,
    c.nationality
)
select
  season,
  constructor_id,
  constructor_name,
  nationality,
  race_starts,
  total_points,
  number_of_wins,
  number_of_podiums,
  RANK() OVER (PARTITION BY season ORDER BY total_points DESC, number_of_wins DESC) AS standing
from
  constructor_session_summary